# 朴素贝叶斯：用概率公式做分类的老兵
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/分类 | 源文件：朴素贝叶斯.py | 核心功能：4 种朴素贝叶斯变体（Gaussian/Multinomial/Bernoulli/Complement）的对比

## 概述

朴素贝叶斯（Naive Bayes）基于贝叶斯定理加上一个"朴素"的假设：**所有特征在给定类别下条件独立**。这个假设在现实中几乎不成立，但朴素贝叶斯的分类效果出奇地好——尤其在文本分类领域，它是经典基线模型。

为什么独立假设不成立却依然有效？因为分类只需要**排序**（哪个类别的后验概率最大），不需要精确的概率值。即使概率估计不准确，排序可能依然正确。

脚本演示了 GaussianNB（连续特征）、MultinomialNB（文本计数）、BernoulliNB（二值特征）和 ComplementNB（不平衡文本）四种变体。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

from sklearn.datasets import fetch_20newsgroups, load_iris
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB
from sklearn.metrics import accuracy_score, classification_report
# --- 导入库 ---
from sklearn.preprocessing import KBinsDiscretizer
import numpy as np

## 数学原理

### 1. 贝叶斯定理与分类

**代码对应**：`GaussianNB().fit(X_train, y_train)` 训练朴素贝叶斯分类器。

贝叶斯分类器基于后验概率：

$$P(y=c|\mathbf{x}) = \frac{P(\mathbf{x}|y=c)P(y=c)}{P(\mathbf{x})} \propto P(\mathbf{x}|y=c)P(y=c)$$

分类规则：$\hat{y} = \arg\max_c P(y=c) \prod_{j=1}^{p} P(x_j|y=c)$

**朴素假设**：给定类别 $c$，各特征条件独立：

$$P(\mathbf{x}|y=c) = \prod_{j=1}^{p} P(x_j|y=c)$$

这个假设在现实中很少成立，但朴素贝叶斯在实践中仍然效果很好——因为分类只需要后验概率的**排序**（$\arg\max$），不需要精确概率值。

### 2. 高斯朴素贝叶斯

**代码对应**：`GaussianNB()` 假设连续特征服从高斯分布。

$$P(x_j|y=c) = \frac{1}{\sqrt{2\pi\sigma_{jc}^2}}\exp\left(-\frac{(x_j - \mu_{jc})^2}{2\sigma_{jc}^2}\right)$$

**代码对应**：`gnb.theta_` 返回各类别的均值 $\mu_{jc}$，`gnb.class_prior_` 返回先验概率 $P(y=c)$。

参数估计（MLE）：

$$\hat{\mu}_{jc} = \frac{1}{n_c}\sum_{i: y_i = c} x_{ij}, \quad \hat{\sigma}_{jc}^2 = \frac{1}{n_c}\sum_{i: y_i = c}(x_{ij} - \hat{\mu}_{jc})^2$$

### 3. 多项式朴素贝叶斯

**代码对应**：`MultinomialNB(alpha=1.0)` 用于文本分类的词频数据。

$$P(x_j|y=c) = \frac{N_{jc} + \alpha}{N_c + \alpha V}$$

其中 $N_{jc} = \sum_{i: y_i = c} x_{ij}$ 为类别 $c$ 中特征 $j$ 的总出现次数，$N_c = \sum_j N_{jc}$，$V$ 为特征数（词表大小）。

**拉普拉斯平滑**（`alpha` 参数）：$\alpha = 1$ 时为拉普拉斯平滑，$\alpha < 1$ 为 Lidstone 平滑。平滑防止某个特征在某个类别中从未出现导致概率为 0（零概率问题：$0$ 乘以任何数都是 $0$，会"抹掉"其他所有特征的信息）。

**代码对应**：代码中 `for alpha in [0.001, ..., 10.0]` 展示了平滑参数的影响。

### 4. 伯努利朴素贝叶斯

**代码对应**：`BernoulliNB(alpha=1.0)` 用于二值特征。

$$P(x_j|y=c) = p_{jc}^{x_j}(1 - p_{jc})^{1 - x_j}$$

其中 $p_{jc}$ 为类别 $c$ 中特征 $j$ 为 1 的概率。

与多项式 NB 的区别：伯努利 NB 显式建模特征**不出现**的概率 $(1 - p_{jc})$，对短文本效果更好。

### 5. 对数空间计算

实际实现中，为避免连乘下溢，使用对数：

$$\ln P(y=c|\mathbf{x}) \propto \ln P(y=c) + \sum_{j=1}^{p}\ln P(x_j|y=c)$$

$$\hat{y} = \arg\max_c \left[\ln P(y=c) + \sum_{j=1}^{p}\ln P(x_j|y=c)\right]$$

### 6. 朴素贝叶斯的理论保证

即使朴素假设不成立，朴素贝叶斯的分类效果也可以很好。原因在于：分类正确只需要 $\arg\max$ 的排序正确，而不需要概率值精确。只要特征间的依赖是"对称的"（对各类别的影响方向相同），排序就不会被破坏。

在 $p$ 个特征中，朴素贝叶斯的收敛速度为 $O(\log n)$（相比一般模型的 $O(p/n)$），这意味着它在小样本场景下特别有优势。

## 代码结构

| 变体 | 适用场景 | 核心 API |
|------|----------|----------|
| GaussianNB | 连续特征（如 Iris） | 假设高斯分布 |
| MultinomialNB | 词频/计数特征 | 文本分类经典选择 |
| BernoulliNB | 0/1 二值特征 | 是否出现而非出现次数 |
| ComplementNB | 类别不平衡文本 | 用补集概率修正 |

### 1. 高斯朴素贝叶斯（连续特征）

运行 1. 高斯朴素贝叶斯（连续特征） 的代码，观察算法在该环节的行为。

In [ ]:
print("=== 高斯朴素贝叶斯 (Iris) ===")
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

gnb = GaussianNB()
gnb.fit(X_train, y_train)
print(f"训练准确率: {gnb.score(X_train, y_train):.4f}")
print(f"测试准确率: {gnb.score(X_test, y_test):.4f}")
print(f"各类别先验概率: {dict(zip(iris.target_names, gnb.class_prior_.round(4)))}")
# --- 输出结果 ---
print(f"各类别均值形状: {gnb.theta_.shape}")
print(f"\n分类报告:\n{classification_report(y_test, gnb.predict(X_test), target_names=iris.target_names)}")

### 2. 多项式朴素贝叶斯（离散特征/计数）

运行 2. 多项式朴素贝叶斯（离散特征/计数） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 多项式朴素贝叶斯（文本分类）===")
# 使用小规模文本数据集
categories = ["sci.space", "rec.sport.baseball", "comp.graphics"]
train_data = fetch_20newsgroups(subset="train", categories=categories, random_state=42)
test_data = fetch_20newsgroups(subset="test", categories=categories, random_state=42)

# 词袋模型向量化
vectorizer = CountVectorizer(max_features=5000, stop_words="english")
X_train_counts = vectorizer.fit_transform(train_data.data)
X_test_counts = vectorizer.transform(test_data.data)

mnb = MultinomialNB(alpha=1.0)  # alpha 为拉普拉斯平滑参数
mnb.fit(X_train_counts, train_data.target)
print(f"训练准确率: {mnb.score(X_train_counts, train_data.target):.4f}")
print(f"测试准确率: {mnb.score(X_test_counts, test_data.target):.4f}")
print(f"类别: {train_data.target_names}")

### 3. 拉普拉斯平滑（alpha）

运行 3. 拉普拉斯平滑（alpha） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== alpha 参数（平滑）对多项式 NB 的影响 ===")
for alpha in [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]:
    mnb_a = MultinomialNB(alpha=alpha)
    mnb_a.fit(X_train_counts, train_data.target)
    test_acc = mnb_a.score(X_test_counts, test_data.target)
# --- 输出结果 ---
    print(f"  alpha={alpha:>6}: 测试准确率={test_acc:.4f}")

### 4. 伯努利朴素贝叶斯（二值特征）

运行 4. 伯努利朴素贝叶斯（二值特征） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 伯努利朴素贝叶斯 ===")
# 将词频转为 0/1（是否出现）
X_train_bin = (X_train_counts > 0).astype(int)
X_test_bin = (X_test_counts > 0).astype(int)

bnb = BernoulliNB(alpha=1.0)
bnb.fit(X_train_bin, train_data.target)
print(f"测试准确率: {bnb.score(X_test_bin, test_data.target):.4f}")

### 5. ComplementNB（互补朴素贝叶斯）

运行 5. ComplementNB（互补朴素贝叶斯） 的代码，观察算法在该环节的行为。

In [ ]:
# 针对类别不平衡优化的 MultinomialNB 变体
print("\n=== ComplementNB（类别不平衡场景）===")
cnb = ComplementNB(alpha=1.0)
cnb.fit(X_train_counts, train_data.target)
print(f"测试准确率: {cnb.score(X_test_counts, test_data.target):.4f}")

### 6. 各朴素贝叶斯变体对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== 各变体在文本分类上的对比 ===")
models = {
    "MultinomialNB": MultinomialNB(alpha=1.0),
    "BernoulliNB": BernoulliNB(alpha=1.0),
    "ComplementNB": ComplementNB(alpha=1.0),
# --- 继续 ---
}
for name, model in models.items():
    if name == "BernoulliNB":
        model.fit(X_train_bin, train_data.target)
        acc = model.score(X_test_bin, test_data.target)
# --- 继续 ---
    else:
        model.fit(X_train_counts, train_data.target)
        acc = model.score(X_test_counts, test_data.target)
    print(f"  {name:>15}: 测试准确率={acc:.4f}")

print("\n=== 朴素贝叶斯要点 ===")
print("- 核心假设：各特征在给定类别下条件独立（现实中很少完全满足，但效果仍好）")
print("- GaussianNB: 适合连续特征（假设特征服从高斯分布）")
print("- MultinomialNB: 适合离散计数数据（文本分类的经典选择）")
print("- BernoulliNB: 适合二值特征（词是否出现，而非词频）")
# --- 输出结果 ---
print("- ComplementNB: 适合类别不平衡的文本分类")
print("- 优点：训练快、小样本友好、可处理多分类")
print("- 缺点：独立假设在强相关特征上表现差")

## 关键代码解释

### 拉普拉斯平滑（alpha）

```python
mnb = MultinomialNB(alpha=1.0)
```

alpha 控制平滑强度。如果不做平滑，某个类别中从未出现过的词会导致整个后验概率为 0（乘法中一个为 0 则全为 0）。拉普拉斯平滑给每个词频加 alpha，确保没有零概率。alpha=1 是标准拉普拉斯平滑，alpha < 1 是 Lidstone 平滑。

### 伯努利 vs 多项式

```python
X_train_bin = (X_train_counts > 0).astype(int)
bnb = BernoulliNB(alpha=1.0)
```

BernoulliNB 只关心词"是否出现"（0/1），不关心出现几次。对于短文本（如推文），BernoulliNB 有时比 MultinomialNB 更好。

## 使用示例

```python
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)
clf = MultinomialNB(alpha=1.0)
clf.fit(X, labels)
```

## 注意事项

1. **特征独立假设**：强相关的特征（如"机器"和"学习"经常一起出现）会导致概率被重复计算
2. **不需要特征缩放**：朴素贝叶斯不基于距离计算
3. **小样本友好**：即使训练数据很少也能给出合理预测
4. **alpha 调参**：通常在 [0.01, 10] 的对数网格上用交叉验证搜索
5. **ComplementNB 专门优化不平衡**：在类别不平衡的文本分类中通常优于 MultinomialNB

## 总结与延伸

以上代码展示了 **朴素贝叶斯** 的完整流程。

**解读要点**：
- 关注 **准确率（Accuracy）** 和 **分类报告** 中的精确率/召回率/F1
- 如果训练准确率远高于测试准确率，说明存在过拟合
- 观察 **混淆矩阵**，找出模型最容易混淆的类别

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **贝叶斯网络**：放松独立假设，建模特征之间的依赖关系
- **TF-IDF + 朴素贝叶斯**：比原始词频效果更好
- **半监督朴素贝叶斯**：利用未标记数据估计特征分布，提升小样本场景的性能
- **在线学习**：partial_fit 支持增量更新，适合流式文本数据
